Data Normalization

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

#Normalize the positional data (GK_x, GK_y, Ball_x, Ball_y)

# Load your training data 
train_df = pd.read_csv('../output/train/train_time_series.csv')

scaler = MinMaxScaler(feature_range=(0, 1))
train_df[['GK_x', 'GK_y', 'Ball_x', 'Ball_y']] = scaler.fit_transform(train_df[['GK_x', 'GK_y', 'Ball_x', 'Ball_y']])

# Load valid data
valid_df = pd.read_csv('../output/valid/valid_time_series.csv')

scaler = MinMaxScaler(feature_range=(0, 1))
valid_df[['GK_x', 'GK_y', 'Ball_x', 'Ball_y']] = scaler.fit_transform(valid_df[['GK_x', 'GK_y', 'Ball_x', 'Ball_y']])

# load test data
test_df = pd.read_csv('../output/test/test_time_series.csv')

scaler = MinMaxScaler(feature_range=(0, 1))
test_df[['GK_x', 'GK_y', 'Ball_x', 'Ball_y']] = scaler.fit_transform(test_df[['GK_x', 'GK_y', 'Ball_x', 'Ball_y']])



Create Sequences for Time-Series Model

In [ ]:
def create_sequences(df, seq_length=30):
    """
    Create sequences of length 'seq_length' from the dataframe.
    Each sequence will be a set of (x, y) positions for the goalkeeper and ball.
    """
    sequences = []
    targets = []

    for i in range(len(df) - seq_length):
        sequence = df.iloc[i:i+seq_length][['GK_x', 'GK_y', 'Ball_x', 'Ball_y']].values
        target = df.iloc[i+seq_length][['GK_x', 'GK_y', 'Ball_x', 'Ball_y']].values
        sequences.append(sequence)
        targets.append(target)

    return np.array(sequences), np.array(targets)

# Example: Create sequences for training, validation, and test data
X_train, y_train = create_sequences(train_df, seq_length=30)
X_valid, y_valid = create_sequences(valid_df, seq_length=30)
X_test, y_test = create_sequences(test_df, seq_length=30)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Define the LSTM model
model = Sequential()
model.add(LSTM(64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(4))  # Output is 4 values: GK_x, GK_y, Ball_x, Ball_y

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Summary of the model
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_valid, y_valid))


In [ ]:
# Evaluate the model on the test data
test_loss = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")


In [ ]:
# Predict on new data (e.g., a sequence from the test set)
y_pred = model.predict(X_test)

# Visualize the predictions and ground truth (e.g., for the goalkeeper's position)
import matplotlib.pyplot as plt

plt.plot(y_test[:, 0], y_test[:, 1], label='True GK Position')
plt.plot(y_pred[:, 0], y_pred[:, 1], label='Predicted GK Position', linestyle='--')
plt.legend()
plt.show()
